# Tutorial Large Language Models

In this tutorial you will use GPT2, a large language model and predecessor of GPT3 and 4 (the backbone model for ChatGPT) to predict the outcome of a prompt.

You will also use fine-tuning to adapt the model to provide reviews for movies.

It is recommended that you run this Notebook in Google Colab:
https://colab.research.google.com/.
If you use Anaconda, you may have to install Keras-NLP seperately.

## Before you start:

Change your Runtime settings to use a GPU, otherwise the waiting times will dramatically increase.

Go to:

Runtime > **Change Runtime **

Select: **T4 GPU**

# PART 1: Create and use a GPT model

## 1) Install Keras NLP:

In [1]:
!pip install git+https://github.com/keras-team/keras-nlp.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 64.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 44.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 98.3 MB/s eta 0:00:00


## 2) Import required libraries:



In [2]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # or "tensorflow" or "torch"

import keras_nlp
import tensorflow as tf
import keras_core as keras

Using JAX backend.


## 3) Create GPT2 model

Next, you create a GPT2 preprocessor and a model.

The processor will take multiple strings and return multiple outputs.

The model is

In [3]:
# To speed up training and generation, we use preprocessor of length 128
# instead of full length 1024.
preprocessor = keras_nlp.models.GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en",
    sequence_length=128,
)


model = keras_nlp.models.GPT2CausalLM.from_preset(
    "gpt2_base_en", preprocessor=preprocessor
)

1042301/1042301 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
456318/456318 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
497986112/497986112 ━━━━━━━━━━━━━━━━━━━━ 24s 0us/step


## 4) Visualise the architecture of the model

You can visualise the architecture of the model using the .summary method:

In [4]:
model.summary()

Preprocessor: "gpt2_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gpt2_tokenizer (GPT2Tokenizer)                     │                                              50,257 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "gpt2_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃     Param # ┃ Connected to                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │           0 │ -                              │
├───────────────────────────────┼───────────────────────────┼─────────────┼────────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │           0 │ -                              │
├───────────────────────────────┼───────────────────────────┼─────────────┼────────────────────────────────┤
│ gpt2_backbone (GPT2Backbone)  │ (None, None, 768)         │ 124,439,808 │ padding_mask[0][0],            │
│                               │                           │             │ token_ids[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────┼────────────────────────────────┤
│ token_embedding               │ (None, None, 50257)       │  38,597,376 │ gpt2_backbone[0][0]            │
│ (ReversibleEmbedding)         │                           │             │                                │
└───────────────────────────────┴───────────────────────────┴─────────────┴────────────────────────────────┘

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
backbone_layer = model.get_layer("gpt2_backbone")
backbone_layer.summary()

Model: "gpt2_backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃    Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)    │ (None, None)           │          0 │ -                          │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ token_embedding           │ (None, None, 50257)    │ 38,597,376 │ token_ids[0][0]            │
│ (ReversibleEmbedding)     │                        │            │                            │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ position_embedding        │ (None, None, 768)      │    786,432 │ token_embedding[0][0]      │
│ (PositionEmbedding)       │                        │            │                            │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ embeddings_add (Add)      │ (None, None, 768)      │          0 │ token_embedding[0][0],     │
│                           │                        │            │ position_embedding[0][0]   │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ embeddings_dropout        │ (None, None, 768)      │          0 │ embeddings_add[0][0]       │
│ (Dropout)                 │                        │            │                            │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ padding_mask (InputLayer) │ (None, None)           │          0 │ -                          │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_0       │ (None, None, 768)      │  7,087,872 │ embeddings_dropout[0][0],  │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_1       │ (None, None, 768)      │  7,087,872 │ transformer_layer_0[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_2       │ (None, None, 768)      │  7,087,872 │ transformer_layer_1[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_3       │ (None, None, 768)      │  7,087,872 │ transformer_layer_2[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_4       │ (None, None, 768)      │  7,087,872 │ transformer_layer_3[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_5       │ (None, None, 768)      │  7,087,872 │ transformer_layer_4[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_6       │ (None, None, 768)      │  7,087,872 │ transformer_layer_5[0][0], │
│ (TransformerDecoder)      │                        │            │ padding_mask[0][0]         │
├───────────────────────────┼────────────────────────┼────────────┼────────────────────────────┤
│ transformer_layer_7       │ (None, None, 768)      │  7,087,872 │ transformer_layer_6[0][0], │
│ (TransformerDecoder) 

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

 Non-trainable params: 0 (0.00 B)

As you can observe, the model is composed of multiple Transformer layers. These are the transformer layers described in the following paper:
https://arxiv.org/pdf/1706.03762.pdf

## 5) Predict output

Here you can write a prompt and use the network to predict an outcome:

In [6]:
prompt = "I love AI"

output = model.generate(prompt, max_length=100)
print("\nGPT-2 output:\n" + output)


GPT-2 output:
I love AI. It's the best thing ever. I'm not saying this is a bad thing, or that it's going to happen, but I'm just saying that it's something that we have to do, because it is the only thing that we know about the world. We know it exists because it's the only thing we know.

It's not like you're going to be able to predict everything. You're going to have to do it in a way. I


**Exercise: Write several prompts to evaluate the quality of the outputs.**


# PART 2: Fine tuning

The rest of the tutorial focuses on adapting the GPT2 model to provide reviews for movies.

You need to run Part 1 before you run Part 2

## 1) Load the IMDB dataset

The IMDB is a large dataset with 25,000 movies reviews from IMDB. The dataset is also labelled as a positive or negative review (although we will not be using the labels).

You can find more information on the dataset here:
https://keras.io/api/datasets/imdb/
and here:
https://ai.stanford.edu/~amaas/data/sentiment/

In [7]:
import tensorflow_datasets as tfds

reddit_ds = tfds.load("imdb_reviews", split="train", as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete2RPAHL/imdb_reviews-train.tfrecord…

Generating test examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete2RPAHL/imdb_reviews-test.tfrecord*…

Generating unsupervised examples...:   0%|          | 0/50000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete2RPAHL/imdb_reviews-unsupervised.t…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


## 2) See example of review in the dataset

In [8]:
for document, title in reddit_ds:
    print(document.numpy())
    print(title.numpy())
    break

b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."
0


## 3) Create the train set:

In [9]:
train_ds = (
    reddit_ds.map(lambda document, _: document)
    .batch(32)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

## 4) Fine tuning of the model with  new data

The following code fits the model to a new dataset.
We take 500 reviews otherwise the training to make training more efficient.

Number of epochs is 1 to start as training takes quite a long time to complete.

In [10]:
train_ds = train_ds.take(100)
num_epochs = 3

loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate = 1e-5),
    loss=loss,
    weighted_metrics=["accuracy"],
)

model.fit(train_ds, epochs=num_epochs)

Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 134s 1s/step - accuracy: 0.3076 - loss: 3.8037
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.3232 - loss: 3.6362
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.3267 - loss: 3.5800


## 5) Predict output

Generate an output for our **previous** prompt:

In [11]:
prompt = "I love AI"

output = model.generate(prompt, max_length=100)
print("\nGPT-2 output:\n" + output)


GPT-2 output:
I love AI. I love the story. And it has a lot of interesting characters. And the plot is pretty good. The plot is good, but the characters are not very interesting. I love the plot. The story is good, but the characters are not very interesting. I love the plot. But the characters are not very interesting. I love the story. The characters are not very interesting. I love the plot. But the characters are not very interesting. I love the story.


**What happened to the output? Do you get the same output or similar?**

## Continue with the GAN tutorial or with the challenge below.

# Challenge:

Adapt GPT2 to another dataset.

You can choose one from Tensorflow:
https://www.tensorflow.org/datasets/catalog/overview#natural_language_understanding